Movie Recommendation System - Student Version
================================================
A simple content-based movie recommendation system that suggests similar movies
based on genres, cast, crew, keywords, and plot overview.

How it works:
1. Load movie data from CSV files
2. Extract and clean movie features (genres, cast, etc.)
3. Combine all features into text tags
4. Convert text to numbers using CountVectorizer
5. Calculate similarity between movies using cosine similarity
6. Recommend movies with highest similarity scores

Required files:
- tmdb_5000_movies.csv
- tmdb_5000_credits.csv


# Import necessary libraries

In [1]:
import numpy as np 
import pandas as pd
import ast  # For safely evaluating string representations of Python data structures
from sklearn.feature_extraction.text import CountVectorizer  # Convert text to numbers
from sklearn.metrics.pairwise import cosine_similarity  # Calculate similarity between movies
import pickle  # Save processed data for later use


# Load and Prepare data

In [2]:
# Load the two CSV files
credits = pd.read_csv('datasets/tmdb_5000_credits.csv')  # Contains cast and crew information
movies = pd.read_csv('datasets/tmdb_5000_movies.csv')   # Contains movie details

In [3]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   object 
 2   homepage              1712 non-null   object 
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   object 
 5   original_language     4803 non-null   object 
 6   original_title        4803 non-null   object 
 7   overview              4800 non-null   object 
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   object 
 10  production_countries  4803 non-null   object 
 11  release_date          4802 non-null   object 
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   object 
 15  status               

In [4]:
movies.shape

(4803, 20)

In [5]:
credits.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movie_id  4803 non-null   int64 
 1   title     4803 non-null   object
 2   cast      4803 non-null   object
 3   crew      4803 non-null   object
dtypes: int64(1), object(3)
memory usage: 150.2+ KB


In [6]:
credits.head(2)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


In [7]:
movies.head(2)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500


In [8]:
print(f"✅ Loaded {len(movies)} movies and {len(credits)} credit records")

✅ Loaded 4803 movies and 4803 credit records


In [9]:
# Merge the two datasets based on movie title
movies = movies.merge(credits, on='title')
print(f"✅ Successfully merged datasets")

✅ Successfully merged datasets


In [10]:

movies.head(2)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",...,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,285,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


In [11]:
# Keep only the columns we need for recommendations
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]

In [12]:
#check if there are any null values or not
movies.isna().sum()

movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [13]:
# Remove rows with missing data (NaN values)
initial_count = len(movies)
movies.dropna(inplace=True)
print(f"✅ Removed {initial_count - len(movies)} rows with missing data")
print(f"✅ Final dataset: {len(movies)} movies")

✅ Removed 3 rows with missing data
✅ Final dataset: 4806 movies


# Data Preprocessing

In [14]:
"""
    Convert string representation of list to actual list of names.
    Example: '[{"name": "Action"}, {"name": "Drama"}]' -> ['Action', 'Drama']
    
    Args:
        obj (str): String representation of list containing dictionaries
    
    Returns:
        list: List of names extracted from the dictionaries
    """
def convert(obj):
    L = []
    try:
        # Safely convert string to Python object
        for i in ast.literal_eval(obj):
            L.append(i['name'])  # Extract the 'name' field from each dictionary
    except:
        # If there's an error, return empty list
        pass
    return L


In [15]:
"""
    Convert cast information and keep only top 3 actors.
    This limits the cast to main actors to avoid too much noise.
    
    Args:
        obj (str): String representation of cast list
    
    Returns:
        list: List of top 3 actor names
    """
def convert_cast(obj):
    L = []
    counter = 0
    try:
        for i in ast.literal_eval(obj):
            if counter < 3:  # Only take first 3 actors
                L.append(i['name'])
                counter += 1
            else:
                break  # Stop after 3 actors
    except:
        pass
    return L

In [16]:
"""
    Extract director name from crew information.
    We only care about the director as they have major influence on movie style.
    
    Args:
        obj (str): String representation of crew list
    
    Returns:
        list: List containing director name (or empty if not found)
    """
def convert_crew(obj):
    try:
        for i in ast.literal_eval(obj):
            if i['job'] == 'Director':  # Look for the director
                return [i['name']]
    except:
        pass
    return []  # Return empty list if no director found

# Process Movie Features

In [17]:
# Apply conversion functions to extract clean data

In [18]:
print("  Extracting genres...")
movies['genres'] = movies['genres'].apply(convert)

  Extracting genres...


In [19]:
print("   Extracting keywords...")
movies['keywords'] = movies['keywords'].apply(convert)

   Extracting keywords...


In [20]:
print("   Extracting cast (top 3)...")
movies['cast'] = movies['cast'].apply(convert_cast)

   Extracting cast (top 3)...


In [21]:
print("   Extracting directors...")
movies['crew'] = movies['crew'].apply(convert_crew)

   Extracting directors...


In [22]:
print("   Processing movie overviews...")
# Split overview text into individual words
movies['overview'] = movies['overview'].apply(lambda x: x.split())

   Processing movie overviews...


# Clean The Text Data

In [23]:
# Remove spaces from names to avoid counting "Tom Cruise" and "TomCruise" as different
print("  Removing spaces from names...")
movies['genres'] = movies['genres'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['cast'] = movies['cast'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['crew'] = movies['crew'].apply(lambda x: [i.replace(" ", "") for i in x])

  Removing spaces from names...


# Combine All Features into Tags

In [24]:
# Combine all features into one list called 'tags'
# This creates a comprehensive description of each movie
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [25]:
# Create a new simplified dataframe with only what we need
new_df = movies[['movie_id', 'title', 'tags']]

In [26]:
new_df.head(2)

,movie_id,title,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d..."


In [27]:
# Convert list of tags to a single string (required for CountVectorizer)
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))

C:\Users\SUMAN MONDAL\AppData\Local\Temp\ipykernel_5508\1071674718.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))


In [28]:
# Convert everything to lowercase for consistency
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

C:\Users\SUMAN MONDAL\AppData\Local\Temp\ipykernel_5508\1925195261.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())


In [29]:
print(" Movie tags created successfully")
print(f" Sample tag: {new_df['tags'].iloc[0][:100]}...")

 Movie tags created successfully
 Sample tag: in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but ...


# Convert Text to Numbers

In [30]:
# Initialize CountVectorizer
# - max_features=5000: Use only top 5000 most common words
# - stop_words='english': Remove common English words like 'the', 'and', etc.
cv = CountVectorizer(max_features=5000, stop_words='english')

In [31]:
# Convert text tags to numerical vectors
# Each movie becomes a vector of 5000 numbers (0s and 1s)
vectors = cv.fit_transform(new_df['tags']).toarray()

In [32]:
print(f" Created vectors of shape: {vectors.shape}")
print(f" {vectors.shape[0]} movies, {vectors.shape[1]}")

 Created vectors of shape: (4806, 5000)
 4806 movies, 5000


# Calculate Movie Similarities

In [33]:
# Calculate cosine similarity between all movies
# This creates a matrix where similarity[i][j] = similarity between movie i and movie j
similarity = cosine_similarity(vectors)

In [34]:
print(f" Similarity matrix created: {similarity.shape}")
print(f"  Matrix contains {similarity.shape[0] * similarity.shape[1]:,} similarity scores")

 Similarity matrix created: (4806, 4806)
  Matrix contains 23,097,636 similarity scores


# Recommendation Function


Recommend similar movies based on content similarity.

movie (str): Name of the movie to base recommendations on
num_recommendations (int): Number of movies to recommend 5

Prints the recommended movies

In [35]:
def recommended(movie):
    
    # Convert input to lowercase for case-insensitive matching
    movie = movie.lower()
    
    # Check if movie exists in our dataset
    if movie not in new_df['title'].str.lower().values:
        print(f"❌ Sorry! '{movie}' not found in our dataset.")
        
        # Try to find similar movie names to help user
        similar_movies = new_df[new_df['title'].str.lower().str.contains(movie.split()[0] if movie.split() else movie, na=False)]
        if not similar_movies.empty:
            print("🔍 Did you mean one of these?")
            for i, title in enumerate(similar_movies['title'].head(5)):
                print(f"   {i+1}. {title}")
        return
    
    # Find the index of the movie in our dataset
    idx = new_df[new_df['title'].str.lower() == movie].index[0]
    
    # Get similarity scores for this movie with all other movies
    # enumerate() gives us (index, similarity_score) pairs
    distances = list(enumerate(similarity[idx]))
    
    # Sort movies by similarity score (highest first)
    # [1:num_recommendations+1] skips the movie itself (index 0) and takes top N
    movies_list = sorted(distances, key=lambda x: x[1], reverse=True)[1:num_recommendations+1]
    
    # Display recommendations
    movie_title = new_df.iloc[idx]['title']
    print(f"\n🎬 Movies similar to '{movie_title}':")
    print("=" * 60)
    
    for i, (movie_idx, score) in enumerate(movies_list, 1):
        recommended_title = new_df.iloc[movie_idx]['title']
        print(f"{i:2d}. {recommended_title:<45} (Similarity: {score:.3f})")
    
    print("=" * 60)

In [36]:
!pip install nltk --quiet

In [37]:
import nltk

In [38]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [39]:
def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

In [40]:
new_df['tags'] = new_df['tags'].apply(stem)

C:\Users\SUMAN MONDAL\AppData\Local\Temp\ipykernel_5508\3213734980.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(stem)


In [41]:
def recommend(movie):
    movie_index = new_df[new_df['title'] == movie].index[0]
    distances = similarity[movie_index]
    movie_list = sorted(list(enumerate(distances)),reverse = True,key=lambda x:x[1])[1:6]
    for i in movie_list:
        print(new_df.iloc[i[0]].title)
        

# Save Processed Data

In [42]:
# Save the processed dataframe and similarity matrix
# This allows us to load them quickly next time without reprocessing
try:
    pickle.dump(new_df, open('movies.pkl', 'wb'))
    pickle.dump(similarity, open('similarity.pkl', 'wb'))
    print("✅ Data saved successfully!")
    print("   📁 movies.pkl - Contains processed movie data")
    print("   📁 similarity.pkl - Contains similarity matrix")
except Exception as e:
    print(f"❌ Error saving data: {e}")

✅ Data saved successfully!
   📁 movies.pkl - Contains processed movie data
   📁 similarity.pkl - Contains similarity matrix


# Demo and Testing

In [43]:
# Test the recommendation system with some popular movies
test_movies = ["Avatar", "The Dark Knight", "Inception", "Titanic"]

In [44]:
for movie in test_movies:
    try:
        recommend(movie)  # Get 3 recommendations for each
        print()  # Add spacing between recommendations
    except:
        print(f"❌ Could not get recommendations for '{movie}'")

Titan A.E.
Small Soldiers
Independence Day
Ender's Game
Aliens vs Predator: Requiem

The Dark Knight Rises
Batman Begins
Batman Returns
Batman Forever
Batman & Robin

Duplex
The Helix... Loaded
Star Trek II: The Wrath of Khan
Timecop
Chicago Overcoat

Raise the Titanic
Captain Phillips
The Notebook
In the Heart of the Sea
Ghost Ship



In [45]:
# For Saving the compressed model
np.savez_compressed("similarity_matrix.npz", similarity=similarity)

# Load later
loaded = np.load("similarity_matrix.npz")
similarity = loaded['similarity']